# Scalable processing and integrations

## Goal

This walkthrough demonstrates bounded chunk processing, ordered thread parallelism, incremental CSV writing, local and in-memory storage adapters, a leakage-safe model pipeline, and local experiment tracking. All files are temporary.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from autoprepml import (
    InMemoryStorageAdapter, LocalExperimentTracker, LocalStorageAdapter,
    make_model_pipeline, process_chunks, stream_process, write_stream,
)

rng = np.random.default_rng(42)
frame = pd.DataFrame({
    "age": rng.integers(18, 75, size=120),
    "balance": rng.normal(500, 120, size=120),
    "segment": rng.choice(["standard", "premium"], size=120),
})
frame["target"] = (frame["balance"] > 500).astype(int)
frame.loc[[4, 22, 90], "balance"] = np.nan
frame.head()

,age,balance,segment,target
0,23,396.100266,standard,0
1,62,616.193403,premium,1
2,55,298.055627,standard,0
3,43,459.813796,premium,0
4,42,NaN,standard,1


### Process bounded chunks and stream the output

In [2]:
def fill_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    result = chunk.copy()
    result["balance"] = result["balance"].fillna(result["balance"].median())
    return result

with TemporaryDirectory(prefix="autoprepml-pipeline-") as temporary_root:
    root = Path(temporary_root)
    source_path = root / "input.csv"
    output_path = root / "cleaned.csv"
    LocalStorageAdapter().write(frame, source_path)

    cleaned = process_chunks(source_path, fill_chunk, chunksize=25, n_jobs=2, backend="thread")
    written_path = write_stream(
        source_path, output_path, fill_chunk, chunksize=25, n_jobs=2, backend="thread"
    )
    print("chunked_shape:", cleaned.shape)
    print("streamed_rows:", len(LocalStorageAdapter().read(written_path)))

    memory = InMemoryStorageAdapter()
    memory.write(cleaned, "cleaned")
    print("in_memory_rows:", len(memory.read("cleaned")))

chunked_shape: (120, 4)
streamed_rows: 120
in_memory_rows: 120


### Compose preprocessing with a model and record an experiment

In [3]:
model = make_model_pipeline(
    cleaned, LogisticRegression(max_iter=300, random_state=42), target_col="target"
)
model.fit(cleaned.drop(columns="target"), cleaned["target"])
predictions = model.predict(cleaned.drop(columns="target").head(8))
print("sample_predictions:", predictions.tolist())

with TemporaryDirectory(prefix="autoprepml-runs-") as run_root:
    tracker = LocalExperimentTracker(run_root)
    with tracker.start_run("chunked-logistic", tags={"backend": "thread"}) as run:
        run.log_params({"chunksize": 25, "workers": 2})
        run.log_metrics({"rows": len(cleaned), "missing_after": float(cleaned.isna().sum().sum())})
        artifact = Path(run_root) / "summary.txt"
        artifact.write_text("chunked workflow complete\n", encoding="utf-8")
        run.log_artifact(artifact)
    tracked_status = tracker.list_runs()[0]["status"]
    print("tracked_runs:", len(tracker.list_runs()))

sample_predictions: [0, 1, 0, 0, 0, 1, 1, 1]
tracked_runs: 1


## Checks

In [4]:
assert len(cleaned) == len(frame)
assert cleaned["balance"].isna().sum() == 0
assert len(predictions) == 8
assert tracked_status == "finished"
print("Scalable pipeline checks passed.")

Scalable pipeline checks passed.
